# Chapter Seven Results

## <font color='green'>Application: Get the secured overnight financing rates for the fourth quarter of 2019.</font>








1.   **Import pandas_datareader**
2.   **Identify Key**: Use search bar with 'secured overnight financing rate'
3.   **Assign dates:**
      * start_date assigned string '2019-10-01' could also be a date or datetime
      * end_date assigned string '2019-12-31' could also be a date or datetime
4.  **Pass arguments to pdr.get_data_fred:**
      * series_key
      * start_date
      * end date



In [1]:
try:
    import pandas_datareader as pdr
except:
    !pip install pandas-datareader
    import pandas_datareader as pdr

In [2]:
series_key='SOFR'
start_date='2019-10-01'
end_date='2019-12-31'
pdr.get_data_fred(series_key,start_date,end_date)

,SOFR
DATE,
2019-10-01,1.88
2019-10-02,1.85
2019-10-03,1.84
2019-10-04,1.82
2019-10-07,1.83
...,...
2019-12-25,NaN
2019-12-26,1.52
2019-12-27,1.53


## <font color='green'>Application: Calculate duration with different maturities, coupons, and rates.</font>


<div style="
    border-left: 12px solid green;
    line-height: 1.5;
    padding: 15px">
<br>



|Maturity|Coupon|Rate|
|-------|-------|---------|
|&nbsp;&nbsp;&nbsp;January 21$^{st}$ 2030|&nbsp;&nbsp;&nbsp;5|&emsp;4%|
|&nbsp;&nbsp;&nbsp;January 21$^{st}$ 2030|&nbsp;&nbsp;&nbsp;5|&emsp;7%|
|&nbsp;&nbsp;&nbsp;January 21$^{st}$ 2030|&nbsp;&nbsp;&nbsp;2|&emsp;4%|
|&nbsp;&nbsp;&nbsp;January 21$^{st}$ 2040|&nbsp;&nbsp;&nbsp;5|&emsp;7%|
|&nbsp;&nbsp;&nbsp;January 21$^{st}$ 2040|&nbsp;&nbsp;&nbsp;0|&emsp;4%|





</div>

### Import required modules


1.   os and sys required for custom module
2.   pandas for DataFrame creation
3.   datetime and date
4.   Download custom module from DropBox and import <font color='green'>calc_duration</font>





1.   Set settlement date
2.   Make maturities, coupons and rates iterable
3.   Iterate through zip of maturities, coupons and rates to populate duration list using <font color='green'>calc_duration</font>



In [4]:
# Import OS to interact with local computer operating system
import os
import sys
import requests
import pandas as pd
from types import ModuleType
# Import the datetime and date classes from the datetime module for working with dates.
from datetime import  date

In [10]:
# Define the URL of the Python module to be downloaded from Dropbox.
# The 'dl=1' parameter in the URL forces a direct download of the file content.
url= 'https://www.dropbox.com/scl/fi/4y5hjxlfphh1ngvbgo77q/\
module_-basic_concepts_fixed_income.py?rlkey=6oxi7mgka42veaat79hcv8boz&st=87sztshr&dl=1'
module_name='basic_concepts_fixed_income'
# Send an HTTP GET request to the URL and store the server's response.
try:
  response=requests.get(url)
  # Raise an exception for bad status codes (like 404 Not Found)
  response.raise_for_status()
  module= ModuleType(module_name)
  #Code contained in response.text executed
  exec(response.text, module.__dict__)
  # Module added to sys
  sys.modules[module_name]=module
except requests.exceptions.RequestException as e:
    print(f"❌ Error: Could not fetch module from URL. {e}")
except Exception as e:
    print(f"❌ Error: Failed to execute or import the module. {e}")
# Open the local file in "write binary" ('wb') mode and save the downloaded content.
# Using a 'with' statement ensures the file is properly closed after writing.

# Now that 'basic_concepts_fixed_income' exists in the notebook, import the specific functions
from basic_concepts_fixed_income import (calc_duration,
                                         bond_pay_data,
                                         bond_pv)

In [6]:
settlement=date(2025,1,21)
#create lists for maturities, coupons and rates
maturities=[date(2030,1,21),date(2030,1,21),date(2030,1,21),date(2040,1,21),date(2040,1,21)]
coupons=[5,5,2,5,0]
rates=[0.04,0.07,0.04,0.07,0.04]
#initialize duration list
durations=[]
for maturity,coupon,rate in zip(maturities,coupons,rates):
  duration=calc_duration(maturity,coupon,rates=rate,settlement=settlement)
  durations.append(duration)

In [7]:
#create DataFrame from dictionary of variables
dict={"Maturity":maturities,"Coupon":coupons,"Rate":rates,"Duration":durations}
df_durations=pd.DataFrame(dict)
df_durations

,Maturity,Coupon,Rate,Duration
0,2030-01-21,5,0.04,4.499497
1,2030-01-21,5,0.07,4.456870
2,2030-01-21,2,0.04,4.771243
3,2040-01-21,5,0.07,10.167858
4,2040-01-21,0,0.04,15.003422


## <font color='green'>Application: Calculate Duration with Minimum and Maximum Par Yields</font>


<div style="
    border-left: 12px solid green;
    line-height: 1.5;
    padding: 15px">

<br>

**Settlement date is December 31$^{st}$ 2025**


|Maturity|Minimum Coupon|Maximum Coupon|Rate|
|-------|-------|---------|-------|
|&nbsp;&nbsp;&nbsp;June 30$^{th}$ 2026|&nbsp;&nbsp;&nbsp;0.02|&nbsp;&nbsp;&nbsp;8.49|&emsp;4.50%|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2035|&nbsp;&nbsp;&nbsp;0.52|&nbsp;&nbsp;&nbsp;9.09|&emsp;4.75%||
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2055|&nbsp;&nbsp;&nbsp;0.99|&nbsp;&nbsp;&nbsp;9.18|&emsp;5.00%|


</div>



1.   Set settlement date
2.   Make maturities, coupons and rates iterable
3.   Iterate through zipped variables calculating duration with <font color='green'>calc_duration</font>


In [8]:
settlement=date(2025,12,31)
#create lists for maturities, coupons and rates
maturities=[date(2026,6,30),date(2035,12,31),date(2055,12,31)]
min_coupons=[.02,.52,.99]
max_coupons=[8.49,9.09,9.18]
rates=[0.045,0.0475,0.05]

#calculate durations for minimum par yields
min_durations=[]
for maturity,coupon,rate in zip(maturities,min_coupons,rates):
  duration=calc_duration(maturity,coupon,rates=rate,settlement=settlement)
  min_durations.append(duration)

  #calculate durations for maximum par yields
max_durations=[]
for maturity,coupon,rate in zip(maturities,max_coupons,rates):
  duration=calc_duration(maturity,coupon,rates=rate,settlement=settlement)
  max_durations.append(duration)

In [ ]:
#create DataFrame from dictionary of variables
dict={"Maturity":maturities,"Min Coupon":min_coupons,"Max Coupon":max_coupons,"Rate":rates,"Duration for Min Coupons":min_durations,
      "Duration for Max Coupons":max_durations}
df_durations=pd.DataFrame(dict)
df_durations

,Maturity,Min Coupon,Max Coupon,Rate,Duration for Min Coupons,Duration for Max Coupons
0,2026-06-30,0.02,8.49,0.0450,0.495551,0.495551
1,2035-12-31,0.52,9.09,0.0475,9.681308,7.248543
2,2055-12-31,0.99,9.18,0.0500,22.563666,14.145992


## <font color='green'>Application: Calculate Duration At Different Yields To Maturity</font>

<div style="
    border-left: 12px solid green;
    line-height: 1.5;
    padding: 15px">
<br>


**Settlement date is December 31$^{st}$ 2025**

|Maturity|Coupon|Low Rate|Mid Rate|High Rate|
|-------|-------|---------|-----|-----|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2035|&nbsp;&nbsp;&nbsp;5.00|&emsp;4.0%|&emsp;5.0%|&emsp;6.0%|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2055|&nbsp;&nbsp;&nbsp;5.00|&emsp;4.0%|&emsp;5.0%|&emsp;6.0%|





</div>



1.   Set settlement date
2.   Make maturities iterable
3.   Iterate through maturities calculating duration for low, mid and high rates using <font color='green'>calc_duration</font>


In [9]:
settlement=date(2025,12,31)
#create list for maturities
maturities=[date(2035,12,31),date(2055,12,31)]
coupon=5
#initialize duration lists
durations_low=[]
durations_mid=[]
durations_high=[]
#for each maturity, calculate duration for low, mid and high rate
for maturity in maturities:
  duration=calc_duration(maturity,coupon,rates=.04,settlement=settlement)
  durations_low.append(duration)
  duration=calc_duration(maturity,coupon,rates=.05,settlement=settlement)
  durations_mid.append(duration)
  duration=calc_duration(maturity,coupon,rates=.06,settlement=settlement)
  durations_high.append(duration)

In [11]:
#create DataFrame from dictionary of variables
dict={"Maturity":maturities,"Coupon":coupon,"Duration at 4.0% ":durations_low,"Duration at 5.0% ":durations_mid,"Duration at 6.0% ":durations_high,}
df_durations=pd.DataFrame(dict)
df_durations

,Maturity,Coupon,Duration at 4.0%,Duration at 5.0%,Duration at 6.0%
0,2035-12-31,5,8.076507,7.982796,7.885500
1,2055-12-31,5,16.888064,15.772744,14.674078


## <font color='green'>Exercise: Compare the Actual and Duration Forecasted Rates of Change in Bond Prices</font>

<br>

**Settlement date is January 31$^{st}$ 2026**



|Maturity|Coupon|Initial YTM|Increased YTM|Decreased YTM|
|-------|-------|-----------|------------|--------------|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2035|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.0|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;7%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3%|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2035|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.0|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.25%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;4.75%|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2055|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.0|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;7%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;3%|
|&nbsp;&nbsp;&nbsp;December 31$^{st}$ 2055|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.0|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;5.25%|&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;4.75%|





1.   Assign settlement, coupon, and initial YTM
2.   Calculate duration by iterating through matruties



In [12]:
settlement=date(2026,1,31)
coupon=5
YTM_initial=.05
# create list of maturity dates. Dates are repeated to facilitate 4 rows in the DataFrame
maturities=[date(2035,12,31),date(2035,12,31),date(2055,12,31),date(2055,12,31)]
# add labels for each maturity, one for actual price changes and one for predicted changes
labels=["Actual Changes","Predicted Changes","Actual Changes","Predicted Changes"]
durations=[]
# lists for each percentage price change
change_seven_pcnt=[]
change_three_pcnt=[]
change_5_25_pcnt=[]
change_4_75_pcnt=[]

# iterate over maturities and labels
for maturity, label in zip(maturities,labels):
  # calculate duration of bond at initial yield to maturity
  duration=calc_duration(maturity,coupon,rates=YTM_initial,
                         settlement=settlement)
  #use pay_data function to get payment dates and amounts for the bond
  pay_data={'pay_data':bond_pay_data(maturity,coupon,
                                     settlement=settlement)}
  # calculate original bond price with bond_pv
  # the function returns bond price, 1st derivative, 2nd derivative
  # adding underscores returns only the bond price
  price_initial,_,_=bond_pv(rates=YTM_initial, data_dict=pay_data,
                            settlement=settlement)
# recalculate bond prices for each new YTM
# Actual changes are calculated differently from predicted changes
  if label == "Actual Changes":
    price_7,_,_=bond_pv(rates=.07, data_dict=pay_data,\
                                   settlement=settlement)
    price_3,_,_=bond_pv(rates=.03, data_dict=pay_data,\
                                   settlement=settlement)
    price_5_25,_,_=bond_pv(rates=.0525, data_dict=pay_data,\
                                   settlement=settlement)
    price_4_75,_,_=bond_pv(rates=.0475, data_dict=pay_data,\
                                   settlement=settlement)
    # percent price change calculated
    pcnt_7=(price_7-price_initial)/price_initial*100
    pcnt_3=(price_3-price_initial)/price_initial*100
    pcnt_5_25=(price_5_25-price_initial)/price_initial*100
    pcnt_4_75=(price_4_75-price_initial)/price_initial*100

  else: #Predicted price changes = -duration * yield change
    pcnt_7= -duration*(.07-YTM_initial)*100
    pcnt_3= -duration*(.03-YTM_initial)*100
    pcnt_5_25= -duration*(.0525-YTM_initial)*100
    pcnt_4_75= -duration*(.0475-YTM_initial)*100
  # populate variables
  change_seven_pcnt.append(pcnt_7)
  change_three_pcnt.append(pcnt_3)
  change_5_25_pcnt.append(pcnt_5_25)
  change_4_75_pcnt.append(pcnt_4_75)
  durations.append(duration)

In [13]:
#create DataFrame from dictionary of variables
dict={"Maturity":maturities," ":labels,"Coupon":coupon,"Initial YTM":YTM_initial,"Duration":durations,"% Change for .07 ":change_seven_pcnt,
      "% Change for .03 ":change_three_pcnt,"% Change for .0525 ":change_5_25_pcnt,"% Change for .0475 ":change_4_75_pcnt}
df_durations=pd.DataFrame(dict)
df_durations

,Maturity,,Coupon,Initial YTM,Duration,% Change for .07,% Change for .03,% Change for .0525,% Change for .0475
0,2035-12-31,Actual Changes,5,0.05,7.897922,-14.444135,17.330051,-1.952180,1.997137
1,2035-12-31,Predicted Changes,5,0.05,7.897922,-15.795844,15.795844,-1.974481,1.974481
2,2055-12-31,Actual Changes,5,0.05,15.687871,-25.315466,39.941954,-3.812739,4.036016
3,2055-12-31,Predicted Changes,5,0.05,15.687871,-31.375741,31.375741,-3.921968,3.921968


Duration acts as a good approximation for interest rate risk when changes in yield to maturity are small. The small YTM changes (0.0525 and 0.0475) produce actual bond price changes that are close to those predicted by duration.  The large YTM changes (0.07 and 0.03), however, have actual bond change values that are very different from the predicted changes, particularly for the higher duration bond. The prediction errors are higher for YTM decreases than YTM increases. This non-linear relationshipe is called convexity and is the subject of the next chapter.
